# 2. Data Preprocessing (PySpark)

Notebook ini 100% menggunakan **PySpark** untuk menarik data JSON yang kotor dari MongoDB, membersihkannya (*clean*), melakukan rekayasa fitur (*feature engineering*), dan menyimpannya kembali ke MongoDB.

In [ ]:
import sys
import os

# Menambahkan root folder ke system path
sys.path.append(os.path.abspath('..'))

In [ ]:
from utils.spark_session import get_spark_session, stop_spark
from src.storage.mongo_connector import save_to_mongo, load_from_mongo
from src.preprocessing.cleaner import clean, summary
from src.preprocessing.feature_engineering import build_features
from config.settings import MONGO_RAW_COL, MONGO_CLEAN_COL

### 1. Inisialisasi Spark Session
Menjalankan sel di bawah ini akan memanggil `SparkSession`. Jika berhasil, tidak ada error dan Anda siap memproses Big Data.

In [ ]:
spark = get_spark_session()
spark

### 2. Memuat Data dari MongoDB (*Load Raw Data*)
Data diambil langsung dari MongoDB oleh Spark.

In [ ]:
raw_df = load_from_mongo(MONGO_RAW_COL)
print("Skema Data Mentah (Nested JSON):")
raw_df.printSchema()

In [ ]:
raw_df.show(5)

### 3. Pembersihan Data (*Cleaning*)

In [ ]:
cleaned_df = clean(raw_df)
print("Skema Data Bersih (Flattened):")
cleaned_df.printSchema()
cleaned_df.show(5)

### 4. Rekayasa Fitur (*Feature Engineering*)
Menggunakan *VectorAssembler* dan *StandardScaler* bawaan PySpark MLlib.

In [ ]:
featured_df = build_features(cleaned_df)
summary(featured_df)

featured_df.select("time", "place", "scaled_features").show(5, truncate=False)

### 5. Simpan Hasil ke MongoDB

In [ ]:
save_to_mongo(featured_df, MONGO_CLEAN_COL)
print("Data bersih berhasil disimpan ke MongoDB!")

### 6. Matikan Spark

In [ ]:
stop_spark()